# Xceptional Vision – Cancer Tissue Classification Using Deep Learning

## Stage 1: CT Image Preprocessing & Medically Valid Data Augmentation

This notebook demonstrates the end-to-end clinical preprocessing pipeline for Computed Tomography (CT) scan images:
1. **Dataset Integrity & Format Audit** (Detecting corrupted or unreadable images)
2. **Raw Image Inspection Before Preprocessing**
3. **Noise Reduction via Edge-Preserving Bilateral Filtering**
4. **Contrast Enhancement via CLAHE (Contrast Limited Adaptive Histogram Equalization) on Luminance**
5. **Resolution Standardization to (299, 299) for Xception**
6. **Normalization to [-1.0, 1.0] Matching Xception Pre-training**
7. **Anatomically Constrained Data Augmentation** (Preserving thoracic axial orientation; no vertical flips)

In [ ]:
# 1. Environment Setup & Module Imports
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ensure local src modules are on python path
sys.path.insert(0, os.path.abspath(".."))

from src.preprocessing import (
    load_image,
    verify_dataset,
    remove_noise,
    enhance_contrast,
    resize_image,
    normalize_for_xception,
    denormalize_from_xception,
    preprocess_image,
    display_raw_sample_grid,
    display_preprocessing_comparison,
    XCEPTION_INPUT_SHAPE
)
from src.augmentation import MedicallyValidAugmentor, visualize_augmentation_variations

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version:  {np.__version__}")

### 2. Dataset Verification and Integrity Audit
We audit all 999 CT scan images across `train`, `validation`, and `test` splits to confirm readability and class distribution.

In [ ]:
audit_report = verify_dataset(dataset_root="../dataset")
print(f"Total Valid Images: {audit_report['total_valid_images']}")
print(f"Corrupted Files:    {len(audit_report['corrupted_files'])}")
print("-" * 50)
for split, counts in audit_report["counts"].items():
    print(f"Split: [{split.upper()}] (Total: {sum(counts.values())})")
    for cls_name, count in counts.items():
        print(f"  - {cls_name:25s}: {count:3d} images")

### 3. Raw CT Scans Before Preprocessing
Visualizing raw CT slices from the dataset across all 4 categories: Adenocarcinoma, Large Cell Carcinoma, Normal, and Squamous Cell Carcinoma.

In [ ]:
sample_paths = {
    "adenocarcinoma": "../sample_images/sample_adenocarcinoma.png",
    "large_cell_carcinoma": "../sample_images/sample_large_cell_carcinoma.png",
    "normal": "../sample_images/sample_normal.png",
    "squamous_cell_carcinoma": "../sample_images/sample_squamous_cell_carcinoma.png"
}

display_raw_sample_grid(sample_paths, save_path="../results/graphs/raw_samples_before_preprocessing.png")

### 4. Step-by-Step Preprocessing Pipeline Stages
Comparing the progressive clinical enhancement stages:
1. **Raw CT Scan** (unaltered ground truth)
2. **Denoised** (Bilateral filtering smoothing noise while keeping tumor borders sharp)
3. **Contrast Enhanced** (CLAHE applied to the Luminance channel in LAB color space)
4. **Standardized for Xception** (Resized to 299x299 px and scaled to $[-1.0, 1.0]$)

In [ ]:
display_preprocessing_comparison(sample_paths, save_path="../results/graphs/preprocessing_comparison.png")

### 5. Medically Realistic Training Augmentations
Demonstrating random transformations tailored for axial thoracic CT imaging without anatomical inversion.

In [ ]:
base_adeno = preprocess_image(sample_paths["adenocarcinoma"], return_normalized=False)
visualize_augmentation_variations(
    base_image=base_adeno,
    class_name="adenocarcinoma",
    save_path="../results/graphs/data_augmentation_samples.png",
    num_variations=5
)